# Task 4 — Forecasting Access and Usage (2025–2027)

Because the time series are sparse, this notebook uses **simple, transparent models** and **scenario ranges**:

- **Access target:** `ACC_OWNERSHIP` (Global Findex, national/all)
- **Usage proxy:** `USG_P2P_COUNT` (P2P transaction count; admin series)

Forecasts are produced for `base`, `optimistic`, and `pessimistic` scenarios.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from src.data import load_enriched_unified
from src.forecasting import (
    build_forecast_table,
    forecast_access_account_ownership,
    forecast_usage_p2p_count,
)

PROJECT_DIR = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
df = load_enriched_unified(PROJECT_DIR)

In [ ]:
scenarios = ['base', 'optimistic', 'pessimistic']
results = []
for s in scenarios:
    results.append(forecast_access_account_ownership(df, scenario=s))
    results.append(forecast_usage_p2p_count(df, scenario=s))

fc = build_forecast_table(results)
fc

In [ ]:
# Save forecast table
out_models = PROJECT_DIR / 'models'
out_models.mkdir(parents=True, exist_ok=True)
out_csv = out_models / 'forecast_table.csv'
fc.to_csv(out_csv, index=False)
print('Wrote:', out_csv)

In [ ]:
# Plot Access forecast
acc = fc[fc.indicator_code == 'ACC_OWNERSHIP']
fig_acc = px.line(acc, x='year', y='forecast_value', color='scenario', markers=True,
                  title='Forecast — Account Ownership Rate (Access)')
fig_acc.update_yaxes(title_text='% of adults')
fig_acc

In [ ]:
# Plot Usage proxy forecast
usg = fc[fc.indicator_code == 'USG_P2P_COUNT']
fig_usg = px.line(usg, x='year', y='forecast_value', color='scenario', markers=True,
                  title='Forecast — P2P Transactions (Usage proxy)')
fig_usg.update_yaxes(title_text='transactions')
fig_usg

In [ ]:
# Save figures (requires kaleido)
figures_dir = PROJECT_DIR / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

fig_acc.write_image(str(figures_dir / 'forecast_access.png'), scale=2)
fig_usg.write_image(str(figures_dir / 'forecast_usage_p2p.png'), scale=2)

print('Saved figures to', figures_dir)